# 00 Kaggle Setup And Data Audit

Purpose: execute the 00 kaggle setup and data audit stage.
Required inputs: DATA_ROOT plus artifacts from prior stages when listed.
Expected outputs: audit report: artifacts/audit/.
Fold: FOLD; seed: SEED; all unexecuted results remain `not_run`.


In [ ]:
from pathlib import Path
import os, sys, platform, re, torch

print("=== CHECKING /kaggle/input ===")
input_dir = Path("/kaggle/input")
if input_dir.exists():
    items = list(input_dir.iterdir())
    print(f"Contents of /kaggle/input ({len(items)} items): {[p.name for p in items]}")
    for it in items:
        if it.is_dir():
            print(f"  Folder {it.name} contains {len(list(it.iterdir()))} items")
else:
    print("/kaggle/input does not exist!")

# Robust discovery of DATA_ROOT
DATA_ROOT = None
if os.environ.get('PPGDALIA_ROOT'):
    p = Path(os.environ['PPGDALIA_ROOT'])
    if p.exists():
        DATA_ROOT = p

if DATA_ROOT is None and input_dir.exists():
    for p in input_dir.rglob("*.pkl"):
        if re.search(r'(?<![A-Z0-9])S(1[0-5]|[1-9])(?![A-Z0-9])', p.stem.upper()):
            rel = p.relative_to(input_dir)
            DATA_ROOT = input_dir / rel.parts[0]
            print(f"Auto-discovered DATA_ROOT via subject files: {DATA_ROOT}")
            break

if DATA_ROOT is None:
    _candidates = [
        Path('/kaggle/input/ppg-dalia-dataset'),
        Path('/kaggle/input/ppg-dalia'),
        Path('/kaggle/input/datasets/ameersifat53/ppg-dalia-dataset')
    ]
    DATA_ROOT = next((p for p in _candidates if p.exists() and p.is_dir()), _candidates[0])

# Project-relative output is collected by krun.yaml outputs: artifacts/.
OUTPUT_ROOT = Path(os.environ.get('PPGCVAE_OUTPUT_ROOT', 'artifacts'))
FOLD = 1
SEED = 42
print({'DATA_ROOT': str(DATA_ROOT), 'OUTPUT_ROOT': str(OUTPUT_ROOT), 'device': 'cuda' if torch.cuda.is_available() else 'cpu', 'python': sys.version, 'torch': torch.__version__})
if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Configure DATA_ROOT to the mounted PPG-DaLiA dataset: {DATA_ROOT}')


In [ ]:
# Locate this repository without assuming a Kaggle dataset slug.
repo_candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'Final_IOT_for_AI']
repo_root = next((p for p in repo_candidates if (p / 'src').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Could not locate repository src/ directory; upload/clone the repository first.')
sys.path.insert(0, str(repo_root / 'src'))
from ppg_cvae.data.splits import get_fold
print(get_fold(FOLD))


In [ ]:
import pandas as pd
from ppg_cvae.data.dalia import discover_subject_files, load_subject
from ppg_cvae.utils.io import save_json
subject_files = discover_subject_files(DATA_ROOT)
expected = {f'S{i}' for i in range(1,16)}
rows = []
for path in subject_files:
    rec = load_subject(path)
    rows.append({'subject_id': rec.subject_id, 'ppg_samples': len(rec.ppg),
                 'hr_labels': None if rec.hr is None else len(rec.hr),
                 'expected_windows_s0_0': max(0,(len(rec.ppg)-512)//128+1),
                 'source_file': str(path)})
audit = pd.DataFrame(rows)
assert set(audit.subject_id) == expected, 'Missing subjects: full study requires S1..S15'
assert audit.subject_id.is_unique, 'Duplicate subject IDs'
audit_dir = OUTPUT_ROOT/'audit'
audit_dir.mkdir(parents=True, exist_ok=True)
audit.to_csv(audit_dir/'dataset_audit.csv', index=False)
print(audit.to_string(index=False))
print('Window counts alone do not establish label offset; inspect the dataset readme before step 01.')
